# 2026-09-02_4 线1 第一步：三组 DEG 的功能画像与异同（module 作解释工具）

分析单位 = **命运组**（DM/DAL/AAM）；module 只是描述工具，**不给 module 贴命运标签**。

回答：DM 组基因主要是什么功能？DAL 组是什么？AAM 组是什么？三组功能异同如何（允许交集）？

**比较逻辑（2×2）**：
- **DM vs DAL**（都胚胎建立，差别=老年是否丢失）→ 哪些区域功能对衰老脆弱 / 哪些稳健维持；
- **(DM+DAL) vs AAM**（都维持到老年，差别=建立时机）→ 哪些功能是成年才新获得；
- 三组共有 module → 贯穿命运的核心区域功能。

**指标两套**：pathway 数 + 锚定基因数（=交集基因：既是 Fig2 该命运 DEG、又在该 pathway 某 stage 的 leading edge）。
**标度两套**：原始数 + 组内占比。module 归属纯描述、不设阈值。
**module 排序**：按 fate 共享结构分组（三组共享 → 两组共享 → 单组特有），组内按数量排。
主线 Macular，Peripheral 同管线出全套；另出双方向合并版便于直接比较。

In [ ]:
suppressPackageStartupMessages({library(dplyr); library(tidyr); library(readr); library(stringr)
  library(ggplot2); library(ggrepel); library(patchwork); library(UpSetR)})

dir_d <- "/project/work/2026-09-01_GSEA_test/Data/"
dir_p <- "/project/work/2026-09-01_GSEA_test/Plot/"
dir.create(dir_p, showWarnings = FALSE, recursive = TRUE)

# 配色：沿用 /project/work/color.ipynb 的 cols.subclass.short
cols_fate  <- c(DM='#3C5488', DAL='#E64B35', AAM='#00A087')
cols_onset <- c(`Embryonic-onset`='#3C5488', `Adult-onset (AAM)`='#00A087')
fate_lv <- c("DM","DAL","AAM")

conn <- read_csv(file.path(dir_d,"2026-09-02_2_fate_pathway_connection_all.csv"), show_col_types=FALSE) %>% filter(pass_U1)
# module 映射使用【作者审定版】combined 文件（脚本 3 的词典产物仅为 _draft 草稿）
mm   <- read_csv(file.path(dir_d,"2026-09-02_3_module_map_combined.csv"), show_col_types=FALSE)
stopifnot(!any(grepl("[^\x01-\x7F]", mm$module)))   # 防非 ASCII 字符导致 PDF 渲染异常

# 由审定版派生 per-direction 两列表（pathway, module），保证与主线一致
for (d in c("Macular","Peripheral")) {
  mm %>% filter(direction==paste0(d," direction")) %>% select(pathway, module) %>% arrange(module, pathway) %>%
    write_csv(file.path(dir_d, paste0("2026-09-02_4_module_map_", d, ".csv")))
}

cj   <- conn %>% left_join(mm, by=c("direction","pathway")) %>% mutate(subclass=factor(subclass, fate_lv))
stopifnot(sum(is.na(cj$module))==0)
cat("U1 连接:", nrow(cj), "| module:", n_distinct(cj$module), "\n")

## 追加(2026-09-17)：PanelA 按方向着色版（`_byDirection` 后缀）

把原 `_unifiedLegend` 的 Macular/Peripheral 两张分向点图合并为**单张、不分块**的点图：

- `x=subclass`(DM/DAL/AAM)，`y=module`；同一 (subclass, module) 位置叠画 central+peripheral 两个点（同心，大点先画、小点在上、半透明可辨）。
- **颜色区分方向**（不再区分 subclass）：`cols.tissue.merged=c(central='#6A3D9A', peripheral='#E69F00')`；`shape=21` 以 `fill` 上色；`central=Macular direction`、`peripheral=Peripheral direction`。
- 点大小 = 该方向该格锚定基因数 `n_gene`（两方向共用 size 标度）。
- **module 排序**：按其出现的 DM/DAL/AAM 组数（两方向合并的共享组数）从多到少，同数按总锚定基因数降序（顶部=三组共享，底部=单组）。
- 输出新文件（不覆盖原图）：`Plot/2026-09-02_4_PanelA_AnchorGenes_n_byDirection_v1.pdf`。


In [ ]:
# ============================================================================
# 【独立追加 cell / 脚本，2026-09-17】PanelA 方向扇形版：同一 dot 改为饼(扇形)
#   - 借鉴 2026-09-09_5_matrix_Central_Peripheral_pie 的 scatterpie 画法
#   - 单张图、不分块(no facet)；x=subclass(DM/DAL/AAM), y=module
#   - 每个 (subclass, module) 画 1 个饼，扇区 = central/peripheral 两方向锚定基因数占比
#     方向配色 cols.tissue.merged (central=#6A3D9A, peripheral=#E69F00)
#   - 饼半径按面积编码两方向基因总数: radius ∝ sqrt(total)；size 图例(隐形点承载)
#   - module 排序: 两方向合并出现的 DM/DAL/AAM 组数从多到少, 同数按总基因数降序; 多者置顶
#   - 去掉格内数字标签; 自包含可单独运行; 不覆盖原有 PDF, 输出 _byDirection_pie
# ============================================================================
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(readr); library(ggplot2); library(scatterpie)
})

.dd <- "/project/work/2026-09-01_GSEA_test/Data/"
.dp <- "/project/work/2026-09-01_GSEA_test/Plot/"
.fate_lv <- c("DM","DAL","AAM")

# 方向配色 (central/peripheral)
cols.tissue.merged <- c('#6A3D9A','#E69F00')
names(cols.tissue.merged) <- c('central','peripheral')

.conn <- read_csv(file.path(.dd,"2026-09-02_2_fate_pathway_connection_all.csv"), show_col_types=FALSE) %>% filter(pass_U1)
.mm   <- read_csv(file.path(.dd,"2026-09-02_3_module_map_combined.csv"), show_col_types=FALSE)
.uniq <- function(x) unique(unlist(strsplit(paste(x, collapse=";"), ";")))

# 单方向 profile: subclass x module -> 锚定基因数
.prof_of <- function(.DIR) {
  .conn %>% left_join(.mm, by=c("direction","pathway")) %>%
    filter(direction==.DIR) %>% mutate(subclass=factor(subclass, .fate_lv)) %>%
    group_by(subclass, module) %>%
    summarise(n_gene=length(.uniq(anchor_genes)), .groups="drop")
}

# central = Macular direction, peripheral = Peripheral direction
.d_long <- bind_rows(
  .prof_of("Macular direction")    %>% mutate(tissue="central"),
  .prof_of("Peripheral direction") %>% mutate(tissue="peripheral")
) %>% filter(!is.na(module), n_gene > 0)

# module 排序: 共享组数(两方向合并的 DM/DAL/AAM 数)从多到少, 同数按总基因数降序
.ord <- .d_long %>% group_by(module) %>%
  summarise(n_fate=n_distinct(subclass), tot=sum(n_gene), .groups="drop") %>%
  arrange(desc(n_fate), desc(tot))
.mod_lv <- .ord$module
.n_mod  <- length(.mod_lv)
cat("module 数:", .n_mod, " | 共享组数分布:",
    paste(names(table(.ord$n_fate)), table(.ord$n_fate), sep="x", collapse=" "), "\n")

# 宽表: 每 (subclass, module) 一行, central/peripheral 两列 -> scatterpie 扇区
.d_wide <- .d_long %>%
  select(subclass, module, tissue, n_gene) %>%
  pivot_wider(names_from=tissue, values_from=n_gene, values_fill=0)
for (tt in c("central","peripheral")) if (!tt %in% names(.d_wide)) .d_wide[[tt]] <- 0
.d_wide <- .d_wide %>% mutate(total = central + peripheral) %>% filter(total > 0)

# 半径面积标定 (面积 ∝ total); 数值坐标 + coord_fixed 保正圆
# 列间距 .xgap 放大, 使 3 列的 x 轴标签(DM/DAL/AAM)不拥挤; 半径不超过行距/半列距
.xgap      <- 2.2
.r_max     <- 0.45
.max_total <- max(.d_wide$total)
.d_wide <- .d_wide %>%
  mutate(x = match(as.character(subclass), .fate_lv) * .xgap,  # DM/DAL/AAM 等距展开
         y = match(as.character(module), rev(.mod_lv)),         # rev: 共享组多者 -> 大 y -> 置顶
         radius = .r_max * sqrt(total / .max_total))
.n_ct <- length(.fate_lv)

.brk <- unique(c(1, 5, .max_total)); .brk <- .brk[.brk <= .max_total]

.p <- ggplot() +
  geom_scatterpie(aes(x=x, y=y, r=radius), data=.d_wide,
                  cols=c("central","peripheral"), color=NA) +
  # 隐形点承载右侧 size 图例(面板内不可见), override.aes 恢复灰色 glyph
  geom_point(data=.d_wide, aes(x=x, y=y, size=total), shape=21, colour=NA, fill=NA) +
  scale_fill_manual(values=cols.tissue.merged, name="Direction",
                    breaks=c("central","peripheral"),
                    guide=guide_legend(order=1, override.aes=list(colour="grey20"))) +
  scale_size_continuous(name="Anchor genes (n)", range=c(1.5, 9), breaks=.brk,
                        limits=c(min(.d_wide$total), .max_total),
                        guide=guide_legend(order=2,
                          override.aes=list(shape=21, fill="grey55", colour="grey30"))) +
  scale_x_continuous(breaks=seq_len(.n_ct) * .xgap, labels=.fate_lv) +
  scale_y_continuous(breaks=seq_len(.n_mod), labels=rev(.mod_lv)) +
  coord_fixed(ratio=1,
              xlim=c(.xgap - .r_max - 0.3, .n_ct * .xgap + .r_max + 0.3),
              ylim=c(0.4, .n_mod + 0.6)) +
  labs(x=NULL, y=NULL, title="Anchor genes (n): Central vs Peripheral") +
  theme_bw(base_size=9) +
  theme(panel.grid.minor=element_blank(),
        panel.grid.major=element_line(color="grey92"),
        panel.border=element_rect(color="grey30", fill=NA),
        axis.text.x=element_text(color="black", size=10),
        axis.text.y=element_text(color="black", size=8),
        plot.title=element_text(size=10, face="bold", hjust=0),
        plot.title.position="plot",
        legend.position="right",
        legend.title=element_text(size=8), legend.text=element_text(size=7.5),
        legend.key.size=unit(0.4,"cm"))

# 由 coord_fixed 的数据纵横比推算画布尺寸, 保证正圆
.s   <- 0.32                                    # 每数据单位英寸
.xr  <- (.n_ct * .xgap + .r_max + 0.3) - (.xgap - .r_max - 0.3)
.yr  <- (.n_mod + 0.6) - 0.4
.w   <- .xr * .s + 3.6                          # + y 轴长标签 + 右侧图例
.h   <- .yr * .s + 0.8
ggsave(file.path(.dp, "2026-09-02_4_PanelA_AnchorGenes_n_byDirection_pie.pdf"),
       .p, width=.w, height=.h)
cat(sprintf("已输出: 2026-09-02_4_PanelA_AnchorGenes_n_byDirection_pie.pdf (w=%.2f h=%.2f) total 范围 %d-%d\n",
            .w, .h, min(.d_wide$total), .max_total))
